In [0]:
# Set base landing directory inside Unity Catalog Volume
base_volume = "/Volumes/training_medallion/00_landing/raw_landing_zone"

# Create format-specific landing directories
dbutils.fs.mkdirs(f"{base_volume}/json_landing")
dbutils.fs.mkdirs(f"{base_volume}/csv_landing")
dbutils.fs.mkdirs(f"{base_volume}/log_landing")
dbutils.fs.mkdirs(f"{base_volume}/text_landing")

print("✅ Landing directories successfully created in Unity Catalog Volume!")

✅ Landing directories successfully created in Unity Catalog Volume!


In [0]:
# Generate synthetic JSON records
json_payload = """{"ticket_id": "TCK-101", "customer_id": 1, "issue": "Payment failed", "status": "OPEN"}
{"ticket_id": "TCK-102", "customer_id": 2, "issue": "Account locked", "status": "CLOSED"}"""

# Save to json_landing folder
json_path = f"{base_volume}/json_landing/tickets_batch1.json"
dbutils.fs.put(json_path, json_payload, overwrite=True)
print(f"✅ JSON file saved to: {json_path}")

Wrote 177 bytes.
✅ JSON file saved to: /Volumes/training_medallion/00_landing/raw_landing_zone/json_landing/tickets_batch1.json


In [0]:
# Generate synthetic CSV records
csv_payload = """customer_id,customer_name,region,total_orders
1,Jenna,APAC,2
2,John,EMEA,6
3,Lea,AMER,10"""

# Save to csv_landing folder
csv_path = f"{base_volume}/csv_landing/customers_batch1.csv"
dbutils.fs.put(csv_path, csv_payload, overwrite=True)
print(f"✅ CSV file saved to: {csv_path}")

Wrote 88 bytes.
✅ CSV file saved to: /Volumes/training_medallion/00_landing/raw_landing_zone/csv_landing/customers_batch1.csv


In [0]:
# Generate synthetic server log entries
log_payload = """2026-10-17 10:15:30 INFO User 1 logged in successfully
2026-10-17 10:16:05 ERROR Payment gateway timeout for ticket TCK-101
2026-10-17 10:17:12 WARN High memory usage detected on node 3"""

# Save to log_landing folder
log_path = f"{base_volume}/log_landing/app_server.log"
dbutils.fs.put(log_path, log_payload, overwrite=True)
print(f"✅ Log file saved to: {log_path}")

Wrote 185 bytes.
✅ Log file saved to: /Volumes/training_medallion/00_landing/raw_landing_zone/log_landing/app_server.log


In [0]:
# Generate synthetic support agent transcript
text_payload = """Customer Jenna reported an issue regarding order payment.
Agent Rudy transferred the call to technical support.
Resolution provided and ticket closed."""

# Save to text_landing folder
text_path = f"{base_volume}/text_landing/chat_transcript_101.txt"
dbutils.fs.put(text_path, text_payload, overwrite=True)
print(f"✅ Text file saved to: {text_path}")

Wrote 150 bytes.
✅ Text file saved to: /Volumes/training_medallion/00_landing/raw_landing_zone/text_landing/chat_transcript_101.txt


In [0]:
files = dbutils.fs.ls("/Volumes/training_medallion/00_landing/raw_landing_zone/")
display(files)

path,name,size,modificationTime
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/_checkpoints/,_checkpoints/,0,1793284376263
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/_schemas/,_schemas/,0,1793284376263
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/batch_risk_parameters/,batch_risk_parameters/,0,1793284376263
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/csv_landing/,csv_landing/,0,1793284376263
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/json_landing/,json_landing/,0,1793284376263
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/log_landing/,log_landing/,0,1793284376263
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/meta_checkpoints/,meta_checkpoints/,0,1793284376263
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/meta_schemas/,meta_schemas/,0,1793284376263
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/risk_management/,risk_management/,0,1793284376263
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/schema_demo_landing/,schema_demo_landing/,0,1793284376263


In [0]:
%fs ls '/Volumes/training_medallion/00_landing/raw_landing_zone/_schemas/json_tickets/_schemas/0/'

path,name,size,modificationTime
dbfs:/Volumes/training_medallion/00_landing/raw_landing_zone/_schemas/json_tickets/_schemas/0,0,427,1792940110000


In [0]:
checkpoint_json = f"{base_volume}/_checkpoints/json_tickets"
schema_json = f"{base_volume}/_schemas/json_tickets"

# Read JSON streaming stream using Auto Loader
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_json)
    .load(f"{base_volume}/json_landing")
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_json)
    .trigger(availableNow=True)
    .table("training_medallion.00_landing.bronze_json_tickets")
)

In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name

checkpoint_json = f"{base_volume}/_checkpoints/json_tickets"
schema_json = f"{base_volume}/_schemas/json_tickets"

# 1. Read JSON stream menggunakan Auto Loader
df_json_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_json)
    .load(f"{base_volume}/json_landing")
)

# 2. Transformasi: Tambahkan kolom created_time dan metadata nama file
df_transformed = (
    df_json_stream
    .withColumn("created_time", current_timestamp()) # 👈 Menambahkan timestamp ingesti
)

# 3. Write Stream ke Bronze Delta Table
query = (
    df_transformed.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_json)
    .option("mergeSchema", "true") # Memastikan kolom baru otomatis diakomodasi skema tabel
    .table("training_medallion.00_landing.bronze_json_tickets")
)

In [0]:
# Generate synthetic JSON records
json_payload = """{"ticket_id": "TCK-104", "customer_id": 4, "issue": "Payment failed", "status": "OPEN"}
{"ticket_id": "TCK-105", "customer_id": 5, "issue": "Account locked", "status": "CLOSED"}"""

# Save to json_landing folder
json_path = f"{base_volume}/json_landing/tickets_batch3.json"
dbutils.fs.put(json_path, json_payload, overwrite=True)
print(f"✅ JSON file saved to: {json_path}")

Wrote 177 bytes.
✅ JSON file saved to: /Volumes/training_medallion/00_landing/raw_landing_zone/json_landing/tickets_batch3.json


In [0]:
%sql
SELECT * FROM training_medallion.00_landing.bronze_json_tickets;

customer_id,issue,status,ticket_id,_rescued_data


In [0]:
%sql 
DESCRIBE HISTORY training_medallion.00_landing.bronze_json_tickets

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-10-29T14:33:10.000Z,2067248053951322,rifqiacim@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""8070ef3e-fbcd-4fd0-bfef-a5bc13352c8b"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-809790ac-4d89-49d0-a0df-bfd8e26df944"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-1de1ae6c-1914-470c-a64d-c58044692374""}, statsOnLoad -> false)",null,List(3870454369809626),b51a82a1-81b6-4b57-a154-d8ff32fcc967,1029-141544-rf3qgsl9-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.12.x-aarch64-photon-scala2.13


In [0]:
checkpoint_csv = f"{base_volume}/_checkpoints/csv_customers"
schema_csv = f"{base_volume}/_schemas/csv_customers"

# Read CSV stream with Auto Loader options
df_csv_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true") # CSV-specific option
    .option("cloudFiles.schemaLocation", schema_csv)
    .load(f"{base_volume}/csv_landing")
)

# Append to Bronze Delta Table
(
    df_csv_stream.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_csv)
    .table("training_medallion.00_landing.bronze_csv_customers")
)

In [0]:
checkpoint_log = f"{base_volume}/_checkpoints/server_logs"

df_log_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .load(f"{base_volume}/log_landing")
)

# Append to Bronze Delta Table
(
    df_log_stream.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_log)
    .table("training_medallion.00_landing.bronze_server_logs")
)

In [0]:
checkpoint_text = f"{base_volume}/_checkpoints/chat_text"

# Read unstructured text files
df_text_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .load(f"{base_volume}/text_landing")
)

# Append to Bronze Delta Table
(
    df_text_stream.writeStream
    .format("delta")
    .trigger(availableNow=True)
    .option("checkpointLocation", checkpoint_text)
    .table("training_medallion.01_bronze.bronze_chat_transcripts")
)